In [98]:
#FULL DOMAIN RUN

# code for tracing particles back to SBZ draft (python version 3.10.9) (not optimized with numpy.where)
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import os; import time

start_time = time.time();

#data loading
################################################################################################################################################################################################################
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
parcel=xr.open_dataset(dir+'cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
whereCL=xr.open_dataset(dir+'Project_Algorithms/whereCL_4_0622.nc').load()
whereCL=whereCL.isel(time=slice(0,len(data['time'])))
kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1)

#job array things
################################################################################################################################################################################################################

num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***

job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
if job_id==0: job_id=1
num_parcels=len(parcel['xh']) #total number of parcels
job_range = num_parcels//num_jobs #number of parcels per job 

# Calculate start and end based on job_id
start_job = (job_id - 1) * job_range
end_job = start_job + job_range
if job_id==num_jobs: end_job=num_parcels
print(f'running for parcels {start_job}-{end_job-1}')
parcel=parcel.isel(xh=slice(start_job,end_job)) #slices the lagrangian parcel data
index_adjust=parcel['xh'].values.astype(int)[0]-1 #for correction of parcel index name

running for parcels 0-2082


In [ ]:
# Lagrangian Tracking Algorithm

#Algorithm Steps:
#(1) Find the first time a parcel is above the LFC:
#(2) First check if the parcel ascends (w>=0.1) for another 20 minutes
#(3) If so, find first time, the parcel slows down (w<0.1)
#(4) If that time is when the parcel is above 750m, save it, "forget", and move on to next parcel
#(5) If that time is when the parcel is below 750m, check if it is within 2km of the CL_Max found from the CL Tracking Algorithm
#(6) If the parcel is near the CL, store in, otherwise save it, "forget", and move on to next parcel
#(7) Continue to next parcel

#(Also, if during, traceback, the parcel escapes the x or z boundary, "forget" parcel, and move on)

In [ ]:
#code for tracing particles back to SBZ draft (python version 3.10.9) 
#(1) row each row in pinfo:
#(2) if pinfo[row,3]>pinfo[row,4], then find that particle in data and trace back to when w<0.1 m/s
#(3) then report max value of convergence within 2kmx2km box

#initialization
################################################################################################################################################################################################################
w_thresh=1e-1/1000 #1e-4 km/s == 0.1 m/s
CLmaxheight=750
ascend_thresh=0e-1/1000
ascend_percent=1.00 #0.00 0.25 #0.50
ascend=np.argmax(times >= 5) #founds how many timesteps is 5 minutes simulation time (always 1 if data timestep > 10)

starttime= 0 
################################################################################################################################################################################################################

def get_3dtime_data(data,varname,tlev):
    cloud_var=data[varname].isel(time=tlev).values
    return cloud_var 

#--------------Function that collects important data for all parcels at time t
def parcel_info(parcel,data,t,visited_edit): #info for all parcels at single time
    #--------------Finds all parcel locations at time t 
    x=parcel['x'].isel(time=t).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; which_x[which_x==-1]=0 #finds which x layer parcel in
    y=parcel['y'].isel(time=t).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; which_y[which_y==-1]=0 #finds which y layer parcel in
    z=parcel['z'].isel(time=t).values/1000; #zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1; which_z[which_z==-1]=0

    #--------------Saves important data 
    #Only looks at parcels that have not been visited before
    looprange=[ind for ind in range(len(parcel['xh'])) if ind not in visited_edit]
    #Initialize storage array 
    pinfo=np.zeros((len(parcel['xh']),6),dtype='object') #num,x,y,z, LFC, LCL
    #Runs through each indicies
    for ind in looprange: 
        #Find LFC levels 
        lfc=data['lfc'].isel(time=t,xh=which_x[ind],yh=which_y[ind]).values/1000;
        lcl=data['lcl'].isel(time=t,xh=which_x[ind],yh=which_y[ind]).values/1000;
        
        # #If LFC is a NAN value, use average of surrounding 10x10 box of LFC values (-don't use-)
        # if lfc<0:
        #     # print(f'no lfc value available for parcel {num} at time {t}. using neighboring lfc average.')
        #     xrange=np.mod(np.arange(which_x[ind]-10,which_x[ind]+10),len(data['xh'])); yrange=np.mod(np.arange(which_y[ind]-10,which_y[ind]+10),len(data['yh'])); 
        #     lfc=data['lfc'].isel(time=t,xh=xrange,yh=yrange).mean().values/1000
        # #If LCL is a NAN value, use average of surrounding 10x10 box of LFC values
        # if lcl<0:
        #     # print(f'no lcl value available at time {t} for parcel {num}. using neighboring lfc.')
        #     xrange=np.mod(np.arange(which_x[ind]-10,which_x[ind]+10),len(data['xh'])); yrange=np.mod(np.arange(which_y[ind]-10,which_y[ind]+10),len(data['yh'])); 
        #     lcl=data['lcl'].isel(time=t,xh=xrange,yh=yrange).mean().values/1000   
        
        #--------------Saves the variables 
        #parcel_num, whichx, whichy, whichz, whichlfc, whichlcl
        pinfo[ind,0]=ind; pinfo[ind,1]=which_x[ind]; 
        pinfo[ind,2]=which_y[ind]; pinfo[ind,3]=z[ind]; pinfo[ind,4]=lfc
        pinfo[ind,5]=lcl #marks tracer number at individual columns
    pinfo=pinfo[~(pinfo == 0).all(axis=1)]
    return(pinfo)

#--------------Function that collects important data for a single parcel_num at time t
def parcel_info_single(parcel,data,t,num): #info for single parcels at single time
     #--------------Finds parcel location at time t 
    x=parcel['x'].isel(time=t,xh=num).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
    if which_x==-1: which_x=0 #finds which x layer parcel in
    y=parcel['y'].isel(time=t,xh=num).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; 
    if which_y==-1: which_y=0 #finds which y layer parcel in
    z=parcel['z'].isel(time=t,xh=num).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1; 
    if which_z==-1: which_z=0
    
    #--------------Saves important data 
    #Gets parcel velocities
    # w=parcel['w'].isel(time=t,xh=num).values/1000; #OLD
    w=data['w'].isel(time=t,xh=which_x,yh=which_y).interp(zf=data['zh']).isel(zh=which_z)/1000 #NEW


    #Find LFC levels
    lfc=data['lfc'].isel(time=t,xh=which_x,yh=which_y).values/1000; 
    lcl=data['lcl'].isel(time=t,xh=which_x,yh=which_y).values/1000;
    # #If LFC is a NAN value, use average of surrounding 10x10 box of LFC values (-don't use-)
    # if lfc<0:
    #     xrange=np.mod(np.arange(which_x-10,which_x+10),len(data['xh'])); yrange=np.mod(np.arange(which_y-10,which_y+10),len(data['yh'])); 
    #     lfc=data['lfc'].isel(time=t,xh=xrange,yh=yrange).mean().values/1000
    # #If LCL is a NAN value, use average of surrounding 10x10 box of LFC values
    # if lcl<0:
    #     xrange=np.mod(np.arange(which_x-10,which_x+10),len(data['xh'])); yrange=np.mod(np.arange(which_y-10,which_y+10),len(data['yh'])); 
    #     lcl=data['lcl'].isel(time=t,xh=xrange,yh=yrange).mean().values/1000   
    
    #--------------Saves the variables 
    #parcel_num, whichx, whichy, whichz, whichlfc, whichlcl
    pinfo_s=np.zeros((1,8+1),dtype='object') #parcel_num,x,y,z, LFC, LCL, u, v, w
    pinfo_s[0,0]=num; pinfo_s[0,1]=which_x; pinfo_s[0,2]=which_y; pinfo_s[0,3]=z; pinfo_s[0,4]=lfc
    pinfo_s[0,5]=lcl; pinfo_s[0,6]=w; pinfo_s[0,7]=which_z; pinfo_s[0,8]=x #marks tracer number,which_x,which_y,z,lfc,lcl,u,v, and w at individual columns
    return(pinfo_s)

################################################################################################################################################################################################################
visited=[] #marks parcel_num that has been visited already
out_arr=np.zeros((len(parcel['xh']),6),dtype='object') #stores parcel_num,x,y,z,time
save_arr=np.zeros((len(parcel['xh']),6),dtype='object') #saves "forgotten" parcels

#1--------------Looping over time
for t in range(starttime,len(whereCL['time'].values)):
    if np.mod(t,10)==0: print(f'current time step: {t}')
    
    #Remove visited parcels from pinfo 
    #And loads parcel_info for all parcels
    ###################################################################################################################################
    visited_edit=[num for num in visited if num in pinfo[:, 0]] #parcel number to delete that are stored in pinfo
    pinfo=parcel_info(parcel,data,t,visited_edit) #array storing information about all parcels
    ###################################################################################################################################
  
    #2--------------Runs through each parcel
    for ind in range(pinfo.shape[0]):

        if pinfo[ind,4] < 0: #new***
            print(f'LFC is negative, forgetting parcel')
            visited.append(pinfo[ind,0]) #adds the parcel to forget to visited to 
            continue
        
        #3--------------If parcel is within 1000m of the LFC, check ascent for several timesteps afterwards
        if (pinfo[ind,4] <= pinfo[ind,3] <= pinfo[ind,4] + 1000/1000) and pinfo[ind,4]>0: #if above LFC (LFC must have a value)
            # above_LFC+=1 #TESTING
            
            #check if parcel continues to ascend for at least 30 minutes simulation time
            #checks if for ascend_percent % of 60 minutes, w is positive
            ascend_range=np.arange(t+1,t+1+ascend,1); ascend_range=ascend_range[ascend_range<len(times)]
            if np.any(ascend_range): ascend_array=parcel['w'].isel(time=ascend_range,xh=pinfo[ind,0]).values/1000
            
            #4--------------If parcel ascends (w>ascend_thresh) for ascend_percent% for ascend timesteps, track back in time
            if np.sum(ascend_array >= ascend_thresh)/len(ascend_array) >= ascend_percent:
                #trace back in time until w<thresh or reach t=0 without w>thresh
                for back_t in np.arange(t-1, starttime-1, -1): #counts from current time down to starttime 
                    
                    #gets info of current parcel
                    pinfo_s=parcel_info_single(parcel,data,back_t,pinfo[ind,0]) 

                    #if current parcel leaves the boundaries, forgets and saves parcel
                    ###################################################################################################################################
                    if pinfo_s[0,8]+pinfo_s[0,6]>np.max(data['xf'].values) or pinfo_s[0,3]+pinfo_s[0,6]>np.max(data['zf'].values): #x is radiative, z non-periodic, forget parcel
                        print(f'parcel {pinfo[ind,0]} crossed x,y,or,z boundary, will forget')
                        visited.append(pinfo[ind,0]) #adds the parcel to forget to visited to delete at next time step
                        break                
                    ###################################################################################################################################

                    #5--------------Tracks when parcel slows down enough to be in gust front (and below 750 m)
                    ###################################################################################################################################
                    #checks if parcel w < w_thresh
                    if pinfo_s[0,6]<w_thresh: 
                        #if parcel is above 750m (or LCL), forget parcel, and move on
                        if pinfo_s[0,3]>=CLmaxheight/1000: 
                            
                            print(f'parcel {pinfo[ind,0]} is above PBL when w < {w_thresh*1000} m/s')
                            #forget parcels that slow down too high up in boundary layer
                            visited.append(pinfo[ind,0]) #adds the parcel to forget to visited to delete at next time step
                            # save_arr[ind,0]=pinfo_s[0,0];save_arr[ind,1]=pinfo_s[0,1];save_arr[ind,2]=pinfo_s[0,2];
                            # save_arr[ind,3]=pinfo_s[0,3];save_arr[ind,4]=back_t;save_arr[ind,5]=t;  #stores parcel num,x,y,z,back_t,t
                            break
                        #6--------------if parcel below 750m (or LCL), check data for max convergence within 2km
                        elif pinfo_s[0,3]<CLmaxheight/1000: #if parcel below 750m (or LCL), check data for max convergence within 2km
                            print(f'parcel {pinfo[ind,0]} w < {w_thresh*1000} m/s at time and below LCL = {back_t}')
                            visited.append(pinfo[ind,0]) #append the visited particle number to delete at next time step #comment if previous append uncommented
                            
                            #Get the data for the x grid location for the CL (at t,z,y)
                            ###################################################################################################################################
                            maxconv_x=whereCL.isel(time=back_t,z=pinfo_s[0,7],y=pinfo_s[0,2])['maxconv_x'].values; #(z level must be < 5)
                            maxconv_x=maxconv_x[maxconv_x>=0] #get rid of non-max nan values (-1)
                            print(f'Current Parcel Info: {pinfo_s}')              
                            ###################################################################################################################################                                              
                            
                            #7--------------Store parcel if parcel is within 2 km of the CL in the x direction
                            ###################################################################################################################################
                            kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1) #finds how many x grids is 1 km
                            tf=np.any(np.isin(maxconv_x, np.arange(pinfo_s[0,1]-2*kms,pinfo_s[0,1]+3*kms)))
                            #checks if the x,y CL indicies match the x,y incidicies of the current parcel
                            if tf==True: #if there is an incidies match with max CL indicies, save parcel
                                print(f'parcel {pinfo[ind,0]} is near CL at t = {back_t}. parcel saved.')
                                out_arr[ind,0]=pinfo_s[0,0];out_arr[ind,1]=pinfo_s[0,1];out_arr[ind,2]=pinfo_s[0,2];
                                out_arr[ind,3]=pinfo_s[0,3];out_arr[ind,4]=back_t;out_arr[ind,5]=t;  #stores parcel num,x,y,z,back_t,t
                                # visited.append(pinfo[ind,0]) #append the visited particle number to delete at next time step #uncomment if previous append commented
                            elif tf==False:
                                #saves parcels that encountered non-CL updraft
                                print(f"parcel number {pinfo_s[0,0]} at time {back_t} saved")
                                save_arr[ind,0]=pinfo_s[0,0];save_arr[ind,1]=pinfo_s[0,1];save_arr[ind,2]=pinfo_s[0,2];
                                save_arr[ind,3]=pinfo_s[0,3];save_arr[ind,4]=back_t;save_arr[ind,5]=t;  #stores parcel num,x,y,z,back_t,t
                            break                
                             ###################################################################################################################################
                    ###################################################################################################################################
    # if np.mod(t,10)==0: print(f'number of parcels above LFC: {above_LFC}') #TESTING
###################################################################################################################################

#Storing output and save data
###################################################################################################################################
out_arr[np.where(np.any(out_arr != 0, axis=1))[0],0]+=index_adjust #*needed for job array*+=index_adjust #*needed for job array*
save_arr[np.where(np.any(save_arr != 0, axis=1))[0],0]+=index_adjust #*needed for job array*+=index_adjust #*needed for job array*
ds=xr.Dataset({
    'out_arr': (['rows', 'columns'], out_arr.astype(float)),
    'save_arr': (['rows', 'columns'], save_arr.astype(float)),
})
ds.to_netcdf(dir+'tracking_algorithms/trackout/parcel_tracking'+str(job_id)+'.nc') #*needed for job array*
# ds.to_netcdf(dir+'tracking_algorithms/trackout/SBZlimited_parcel_tracking'+str(job_id)+'.nc')
###################################################################################################################################
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds")  
#2d5m10,000p30j: 26 minutes
#2d5m125,00030j: max 50-370 minutes (recommended use 500-1000 job arrays for 10 days)

In [ ]:
################################################################
#Run after finishing job array run

In [52]:
#combine all job output arrays 
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import os; import time
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

num_jobs=60 #***
for id in range(1, num_jobs+1):
    # Open the dataset and append it to the list
    if id == 1: 
        out_arr = xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking'+str(id)+'.nc')['out_arr']
        save_arr = xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking'+str(id)+'.nc')['save_arr']
    elif id >= 2: 
        out2 = xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking'+str(id)+'.nc')['out_arr']
        save2 = xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking'+str(id)+'.nc')['save_arr']
        out_arr=np.concatenate((out_arr, out2), axis=0)
        save_arr=np.concatenate((save_arr, save2), axis=0)
        # if np.any(out2): print(id)
ds=xr.Dataset({
    'out_arr': (['rows', 'columns'], out_arr.astype(float)),
    'save_arr': (['rows', 'columns'], save_arr.astype(float)),
})
ds.to_netcdf(dir+'tracking_algorithms/trackout/parcel_tracking_combined.nc')

In [ ]:
# #LIMITED TO SBZ REGION


# #combine all job output arrays 
# import numpy as np
# import matplotlib.pyplot as plt
# import xarray as xr
# import os; import time
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# num_jobs=60 #***
# for id in range(1, num_jobs+1):
#     # Open the dataset and append it to the list
#     if id == 1: 
#         out_arr = xr.open_dataset(dir+'tracking_algorithms/trackout/SBZlimited_parcel_tracking'+str(id)+'.nc')['out_arr']
#         save_arr = xr.open_dataset(dir+'tracking_algorithms/trackout/SBZlimited_parcel_tracking'+str(id)+'.nc')['save_arr']
#     elif id >= 2: 
#         out2 = xr.open_dataset(dir+'tracking_algorithms/trackout/SBZlimited_parcel_tracking'+str(id)+'.nc')['out_arr']
#         save2 = xr.open_dataset(dir+'tracking_algorithms/trackout/SBZlimited_parcel_tracking'+str(id)+'.nc')['save_arr']
#         out_arr=np.concatenate((out_arr, out2), axis=0)
#         save_arr=np.concatenate((save_arr, save2), axis=0)
#         # if np.any(out2): print(id)
# ds=xr.Dataset({
#     'out_arr': (['rows', 'columns'], out_arr.astype(float)),
#     'save_arr': (['rows', 'columns'], save_arr.astype(float)),
# })
# ds.to_netcdf(dir+'tracking_algorithms/trackout/SBZlimited_parcel_tracking_combined.nc')

In [ ]:
################

In [61]:
##### viewing tracking algorithm results
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time

# check=False
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=data['time']
# parcel=xr.open_dataset(dir+'cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# parcel=parcel.isel(time=times.astype(int)) 
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
parcel=xr.open_dataset(dir+'cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
whereCL=xr.open_dataset(dir+'tracking_algorithms/whereCL_4_0622.nc').load() #***
whereCL=whereCL.isel(time=slice(0,len(data['time'])))

# out=xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking4tundra-7_062217.nc')['out_arr'].values;out=out.astype(object);out[:, [0,1,2,4,5]] = out[:, [0,1,2,4,5]].astype(int) #***
# save=xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking4tundra-7_062217.nc')['save_arr'].values;save=save.astype(object);save[:, [0,1,2,4,5]] = save[:, [0,1,2,4,5]].astype(int) #***

out=xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking_combined.nc')['out_arr'].values;out=out.astype(object);out[:, [0,1,2,4,5]] = out[:, [0,1,2,4,5]].astype(int) #***
save=xr.open_dataset(dir+'tracking_algorithms/trackout/parcel_tracking_combined.nc')['save_arr'].values;save=save.astype(object);save[:, [0,1,2,4,5]] = save[:, [0,1,2,4,5]].astype(int) #***

out_nz=out[~np.all(out == 0, axis=1)];print('list of first 10 SBZ parcels'); print(out_nz[:15])
save_nz=save[~np.all(save == 0, axis=1)];save_nz=save_nz[np.where(np.unique(save_nz[1:-1,0]))];print('list of first 10 ignored parcels');print(save_nz[:5])

###############################################################################
#remove duplicates
lst=[]
unique_values, counts = np.unique(out_nz[:,0], return_counts=True); duplicates = unique_values[counts > 1]
for elem in duplicates:
    idx = np.where(out_nz[:,0] == elem)[0] 
    extras=idx[np.where(out_nz[idx,5]!=np.min(out_nz[idx,5]))]
    lst.extend([x for x in extras])
mask=np.ones(len(out_nz), dtype=bool); mask[lst] = False
out_nz=out_nz[mask]; 
placeholder=out_nz.copy(); run=True
###############################################################################
print(f'there are a total of {len(out_nz)} SBZ parcels and {len(save_nz)} non-SBZ parcels')

# #################################################################################################################################
# if check==True:
#     list=[]
#     print('checking if all found SBZ parcels originate from CL')
#     for ind in range(0,out_nz.shape[0]):
#         x=parcel['x'].isel(time=out_nz[ind,4],xh=out_nz[ind,0]).values/1000;xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1 
#         y=parcel['y'].isel(time=out_nz[ind,4],xh=out_nz[ind,0]).values/1000;yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1 
#         # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_x=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
#         # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_y=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
#         z=parcel['z'].isel(time=out_nz[ind,4],xh=out_nz[ind,0]).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1;
#         maxconv_x=whereCL['maxconv_x'].isel(time=out_nz[ind,4],y=which_y,z=which_z).values
#         list.append(any(np.isin(maxconv_x, np.arange(which_x-2,which_x+3))))
#     print(list[:20])
#     list=[]
#     print('checking if all found SBZ parcels hit LFC')
#     for ind in range(0,out_nz.shape[0]):
#         xloc=parcel['x'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000; yloc=parcel['y'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
#         xf=data['xf'].values; which_x=np.searchsorted(xf,xloc)-1; yf=data['yf'].values; which_y=np.searchsorted(yf,yloc)-1; 
#         # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_x=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
#         # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_y=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
#         z=parcel['z'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000 
#         lfc=data['lfc'].isel(time=out_nz[ind,5],xh=which_x,yh=which_y).values/1000
#         list.append(z>=lfc)
#     print(list[:20])
# #################################################################################################################################

# out_min=np.round(np.min(out_nz[:,3]),3);out_max=np.round(np.max(out_nz[:,3]),3); print(f"CL zlev range is {out_min}:{out_max:.3f} km")
# out_mean=np.round(np.mean(out_nz[:,3]),3); print(f'mean CL zlev is {out_mean} km')


list of first 10 SBZ parcels
[[38 311 51 0.02183761978149414 17 25]
 [160 463 51 0.06306727600097656 20 24]
 [244 290 99 0.3974501647949219 10 14]
 [363 455 57 0.13430926513671876 45 52]
 [414 392 93 0.30103671264648435 8 11]
 [452 334 53 0.1520338897705078 43 48]
 [548 368 75 0.19446072387695312 55 60]
 [704 374 63 0.24067965698242189 57 62]
 [746 382 59 0.19366136169433593 8 12]
 [750 305 7 0.11544960784912109 24 29]
 [761 370 93 0.292569091796875 45 49]
 [822 421 1 0.09291950225830078 22 27]
 [838 444 44 0.09400279998779297 49 55]
 [877 275 74 0.36305859375 20 26]
 [874 333 40 0.13773808288574219 44 49]]
list of first 10 ignored parcels
[[34 302 76 0.7418956298828125 34 37]
 [246 338 77 0.33685012817382814 2 11]
 [365 357 3 0.658669677734375 115 121]
 [363 440 62 0.18019241333007813 6 14]
 [483 465 43 0.2835696716308594 27 33]]
there are a total of 1861 SBZ parcels and 1717 non-SBZ parcels


In [62]:
save_nz

array([[34, 302, 76, 0.7418956298828125, 34, 37],
       [246, 338, 77, 0.33685012817382814, 2, 11],
       [365, 357, 3, 0.658669677734375, 115, 121],
       ...,
       [109728, 474, 47, 0.10089625549316407, 9, 13],
       [109785, 486, 81, 0.01489809799194336, 47, 51],
       [109987, 329, 90, 0.1959381561279297, 8, 12]], dtype=object)

In [21]:
# #search for deep convective parcels within lagrangian tracking output     
# ##############################################################
# def threshold(zthresh):
#     out_nz=placeholder.copy()
    
#     deep_out_ind=[]; extendrange=[]
#     times=data['time'].values/(1e9 * 60); times=times.astype(float);
#     for ind in range(len(out_nz)): 
#         #searchs if next most local max goes above zthresh
#         nummins=120; numsteps=int(nummins/times[1])
#         aboverange=np.arange(out_nz[ind,5],out_nz[ind,5]+numsteps,1) #range of times between current time and numsteps later
#         aboverange=aboverange[aboverange<len(data['time'])] #caps out at max time
#         above=parcel['z'].isel(xh=out_nz[ind,0],time=aboverange).values/1000
    
#         dt=1
#         #takes dabove/dt
#         f=above
#         ddx = (
#                 f[1:  ]
#                 -
#                 f[0:-1]
#             ) / (
#             2 * dt
#         )
#         signs = np.sign(ddx)
#         signs_diff=np.diff(signs)
#         local_maxes=np.where((signs_diff != 0) & (signs_diff < 0))[0]+1 #make sure +1 is here
#         if len(local_maxes)==0:
#             local_maxes=[0]
        
#         if np.any(above[local_maxes[0]]>zthresh):
#             extendrange.append(local_maxes[0]) #save to extend xlim of plot later
#             deep_out_ind.append(ind)
    
#     out_nz=out_nz[deep_out_ind,:]
#     # print(f'> {zthresh} km. {len(out_nz)} leftover parcels')
#     return out_nz, extendrange
#     # print(out_nz)
# ##############################################################

# [out_nz,extendrange]=threshold(6)
# print(out_nz)
# print(f'there are a total of {len(out_nz)} SBZ parcels and {len(save_nz)} forgotten parcels')

[[452 334 53 0.1520338897705078 43 48]
 [704 374 63 0.24067965698242189 57 62]
 [885 300 10 0.11673685455322266 59 65]
 ...
 [123793 438 68 0.30884375 74 77]
 [123921 332 38 0.26729971313476564 44 49]
 [124293 368 45 0.09691925811767578 49 53]]
there are a total of 216 SBZ parcels and 19240 forgotten parcels


In [24]:
# #Plot all tracked trajectories
# for k in range(len(out_nz)):
#     choose=parcel.isel(xh=out_nz[k,0],time=range(out_nz[k,4],out_nz[k,5]+1))
#     x=choose['x']/1000
#     z=choose['z']/1000
#     plt.plot(x,z)
# plt.xlabel('#')
# plt.ylabel('z (km)')

In [ ]:
#Plotting Selected SBZ Parcel Individual Plots
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
folder_path = '/mnt/lustre/koa/koastore/torri_group/air_directory/tracking_algorithms/trackout/plots/SBZ'
import os; os.makedirs(folder_path, exist_ok=True)
start_time=time.time()
for k in range(len(out_nz)):

    print(f'{k}/{len(out_nz)}')


    
    print(f'parcel number {out_nz[k:k+1,0]}')
    for xh in out_nz[k:k+1,0]:
        #plotting full z trajectory
        z=parcel['z'].isel(xh=xh).values/1000; 
        channel_aspect_ratio = 3
        plt.figure(figsize=(10, 10/channel_aspect_ratio)) 
        plt.plot(range(len(z)),z)

        #coloring line by x location
        lst=[]
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; #xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_x=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster

            lst.append(which_x)
        colors = plt.cm.viridis  # Choose a colormap, here using 'viridis'
        norm = plt.Normalize(vmin=min(lst), vmax=max(lst))
        plt.scatter(range(len(z)), z, c=lst, cmap=colors, norm=norm,s=4)
        plt.colorbar(label='x grid')

        #data for CL parcel location
        ind=np.where(out_nz[:,0]==xh)[0][0]
        z_CL=parcel['z'].isel(time=out_nz[ind][4],xh=xh).values/1000;

        #data for LFC parcel location
        xloc=parcel['x'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000; yloc=parcel['y'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
        xf=data['xf'].values; which_x=np.searchsorted(xf,xloc)-1; yf=data['yf'].values; which_y=np.searchsorted(yf,yloc)-1; 
        # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_x=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
        # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_y=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
        z=parcel['z'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
        z_lfc=data['lfc'].isel(time=out_nz[ind,5],xh=which_x,yh=which_y).values/1000

        #plotting points for CL and LFC parcel location
        plt.scatter(out_nz[ind][4],z_CL,color='red',zorder=10,label='Convergence Max') #plotting max CL point
        plt.scatter(out_nz[ind][5],z,color='orange',zorder=10,label='LFC') #plotting LFC points
        plt.legend()

        #add some text with the velocity at the CL
        x_center = sum(plt.xlim()) / 2
        y_center = sum(plt.ylim()) / 2
        text='CL w: '+str(parcel['w'].isel(time=out_nz[ind][4],xh=xh).values)
        plt.text(x_center, y_center,text, ha='center', va='center')

        #add some text with the velocity at the LFC
        x_center = sum(plt.xlim()) / 3
        y_center = sum(plt.ylim()) / 3
        text='CL w: '+str(parcel['w'].isel(time=out_nz[ind][5],xh=xh).values)
        plt.text(x_center, y_center,text, ha='center', va='center')

        #plotting the line for LFC at all timesteps
        lst=[]
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            y=parcel['y'].isel(time=t,xh=xh).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; 
            # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_xf=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
            # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_yf=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
            lfc=data['lfc'].isel(time=t,xh=which_x,yh=which_y).values/1000
            lst.append(lfc)
        plt.plot(np.arange(0, len(data['time'])), lst,color='purple',linewidth=0.75) #plotting LFC line

        #plotting all times the parcel is near the CL including where it doesn't go above LFC
        whereCL=xr.open_dataset(dir+'tracking_algorithms/whereCL_4_0622.nc')['maxconv_x'] #***
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            y=parcel['y'].isel(time=t,xh=xh).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; 
            # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_xf=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
            # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_yf=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
            z=parcel['z'].isel(time=t,xh=xh).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1; 
            
            if z<750/1000:
                maxconv_x=whereCL.isel(time=t,y=which_y,z=which_z);
                tf=np.any(np.isin(maxconv_x,np.arange(which_x-2,which_x+3)))
                if tf==True:
                    plt.scatter(t,z,color='red') #plotting additional max CL points

    plt.legend()            
    plt.savefig(os.path.join(folder_path, f"parcel_{out_nz[ind,0]}.png")) 
    plt.close()    
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds") #15 minutes for 100 parcels #5 minutes for 20 parcels

In [ ]:
#Plotting Selected non-SBZ Parcel Individual Plots
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
folder_path = '/mnt/lustre/koa/koastore/torri_group/air_directory/tracking_algorithms/trackout/plots/non-SBZ'
import os; os.makedirs(folder_path, exist_ok=True)
start_time=time.time()
for k in range(len(save_nz)):

    print(f'{k}/{len(save_nz)}')


    
    print(f'parcel number {save_nz[k:k+1,0]}')
    for xh in save_nz[k:k+1,0]:
        #plotting full z trajectory
        z=parcel['z'].isel(xh=xh).values/1000; 
        channel_aspect_ratio = 3
        plt.figure(figsize=(10, 10/channel_aspect_ratio)) 
        plt.plot(range(len(z)),z)

        #coloring line by x location
        lst=[]
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; #xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_x=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster

            lst.append(which_x)
        colors = plt.cm.viridis  # Choose a colormap, here using 'viridis'
        norm = plt.Normalize(vmin=min(lst), vmax=max(lst))
        plt.scatter(range(len(z)), z, c=lst, cmap=colors, norm=norm,s=4)
        plt.colorbar(label='x grid')

        #data for CL parcel location
        ind=np.where(save_nz[:,0]==xh)[0][0]
        z_CL=parcel['z'].isel(time=save_nz[ind][4],xh=xh).values/1000;

        #data for LFC parcel location
        xloc=parcel['x'].isel(time=save_nz[ind,5],xh=save_nz[ind,0]).values/1000; yloc=parcel['y'].isel(time=save_nz[ind,5],xh=save_nz[ind,0]).values/1000
        xf=data['xf'].values; which_x=np.searchsorted(xf,xloc)-1; yf=data['yf'].values; which_y=np.searchsorted(yf,yloc)-1; 
        # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_x=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
        # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_y=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
        z=parcel['z'].isel(time=save_nz[ind,5],xh=save_nz[ind,0]).values/1000
        z_lfc=data['lfc'].isel(time=save_nz[ind,5],xh=which_x,yh=which_y).values/1000

        #plotting points for CL and LFC parcel location
        plt.scatter(save_nz[ind][4],z_CL,color='red',zorder=10,label='Convergence Max') #plotting max CL point
        plt.scatter(save_nz[ind][5],z,color='orange',zorder=10,label='LFC') #plotting LFC points
        plt.legend()

        #add some text with the velocity at the CL
        x_center = sum(plt.xlim()) / 2
        y_center = sum(plt.ylim()) / 2
        text='CL w: '+str(parcel['w'].isel(time=save_nz[ind][4],xh=xh).values)
        plt.text(x_center, y_center,text, ha='center', va='center')

        #add some text with the velocity at the LFC
        x_center = sum(plt.xlim()) / 3
        y_center = sum(plt.ylim()) / 3
        text='CL w: '+str(parcel['w'].isel(time=save_nz[ind][5],xh=xh).values)
        plt.text(x_center, y_center,text, ha='center', va='center')

        #plotting the line for LFC at all timesteps
        lst=[]
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            y=parcel['y'].isel(time=t,xh=xh).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; 
            # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_xf=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
            # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_yf=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
            lfc=data['lfc'].isel(time=t,xh=which_x,yh=which_y).values/1000
            lst.append(lfc)
        plt.plot(np.arange(0, len(data['time'])), lst,color='purple',linewidth=0.75) #plotting LFC line

        #plotting all times the parcel is near the CL including where it doesn't go above LFC
        whereCL=xr.open_dataset(dir+'tracking_algorithms/whereCL_4_0622.nc')['maxconv_x'] #***
        for t in range(0,len(data['time'])):
            x=parcel['x'].isel(time=t,xh=xh).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1; 
            y=parcel['y'].isel(time=t,xh=xh).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1; 
            # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_xf=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
            # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_yf=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
            z=parcel['z'].isel(time=t,xh=xh).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1; 
            
            if z<750/1000:
                maxconv_x=whereCL.isel(time=t,y=which_y,z=which_z);
                tf=np.any(np.isin(maxconv_x,np.arange(which_x-2,which_x+3)))
                if tf==True:
                    plt.scatter(t,z,color='red') #plotting additional max CL points

    plt.legend()            
    plt.savefig(os.path.join(folder_path, f"parcel_{save_nz[ind,0]}.png")) 
    plt.close()    
end_time = time.time(); elapsed_time = end_time - start_time; print(f"Total Elapsed Time: {elapsed_time} seconds") #15 minutes for 100 parcels #5 minutes for 20 parcels

In [ ]:
############################################################
#TESTING

In [28]:
# #Plotting multiple plots at once
# parcel=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_pdata_test5.nc')
# parcel=parcel.isel(time=np.arange(0, len(parcel['time']), 30))

# k=0
# a=4*k; b=a+3; num=b-a
# for xh in out_nz[a:b,0]:
#     z=parcel['z'].isel(xh=xh).values/1000; 
#     plt.plot(range(len(z)),z)
    
#     ind=np.where(out_nz[:,0]==xh)[0][0]
#     z_LFC=parcel['z'].isel(time=out_nz[ind][4],xh=xh).values/1000;
    
#     xloc=parcel['x'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000; yloc=parcel['y'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
#     xf=data['xf'].values; which_x=np.searchsorted(xf,xloc)-1; yf=data['yf'].values; which_y=np.searchsorted(yf,yloc)-1; 
#     # xbins=data['xf'].values; dx=xbins[1]-xbins[0]; which_xf=np.floor(x/dx).astype(int)+np.where(data['xf']==0)[0].item() #faster
#     # ybins=data['yf'].values; dy=ybins[1]-ybins[0]; which_yf=np.floor(y/dy).astype(int)+np.where(data['yf']==0)[0].item() #faster
#     z=parcel['z'].isel(time=out_nz[ind,5],xh=out_nz[ind,0]).values/1000
#     z_lfc=data['lfc'].isel(time=out_nz[ind,5],xh=which_x,yh=which_y).values/1000
    
#     plt.scatter(out_nz[ind][4],z_LFC,color='red',zorder=10,label='Convergence Max')
#     plt.scatter(out_nz[ind][5],z,color='purple',zorder=10,label='LFC')
    
# plt.title(f'trajectory, CL+LFC location of plots {a}:{b} SBZ parcels')
# plt.xlabel('timestep')
# plt.ylabel('z level (m)')

# handles, labels = plt.gca().get_legend_handles_labels()
# plt.legend(handles[num-1:-(num-1)], labels[num-1:-(num-1)])

In [30]:
# #some lagrangian parcel trajectories plots
# def parcel_analysis(ind,var): #ind is number SBZ parcel, var is the variable
#     time_range=np.arange(out_nz[ind,4],out_nz[ind,5]+1)
#     analysis=parcel[var].isel(time=time_range,xh=out_nz[ind,0]).values
#     return analysis

# fig, axs = plt.subplots(2, 2, figsize=(12, 8))  # 2x2 subplots

# variables=['w', 'b', 'qv', 'qc']  
# titles = ['w', 'buoyancy', 'qv', 'qc']  
# ylabs=['m/s', 'm/s/s', 'kg/kg', 'kg/kg']

# for i, var in enumerate(variables):
#     row = i // 2
#     col = i % 2
#     for which in range(len(out_nz)):
#         axs[row, col].plot(parcel_analysis(which, var))
#     axs[row, col].set_title(titles[i])
#     axs[row, col].set_xlabel('time (5 min intervals)')
#     axs[row, col].set_ylabel(ylabs[i]) 

# plt.tight_layout()

# # plt.close()
# # for which in range(0,len(out_nz)):
# #     plt.plot(parcel_analysis(which,'qc')+parcel_analysis(which,'qr'))
# # plt.title('ql (qc+qr)') 
# # plt.xlabel('time (5 min intervals)') 



In [31]:
# fig = plt.figure()

# plt1 = fig.add_subplot(2, 1, 1)  # 2 rows, 1 column, subplot 1
# plt2 = fig.add_subplot(2, 1, 2)  # 2 rows, 1 column, subplot 2

# # Plot data on the first subplot
# for xh in out_nz[:,0]:
#     z = parcel['z'].isel(xh=xh).values / 1000
#     plt1.plot(z)

# plt1.set_title('SBZ parcels that went from near CL to LFC') 
# plt1.set_xlabel('timestep')
# plt1.set_ylabel('z level (m)')

# # Plot data on the second subplot
# for xh in save_nz[:,0]:
#     z = parcel['z'].isel(xh=xh).values / 1000
#     plt2.plot(z)

# plt2.set_title('parcels that did NOT go from near CL to LFC') 
# plt2.set_xlabel('timestep')
# plt2.set_ylabel('z level (m)')

# plt.tight_layout()  # Adjust layout to prevent overlap

# plt.show()

In [262]:
# parcel=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_pdata_test5.nc')
# parcel=parcel.isel(time=np.arange(0, len(parcel['time']), 30))

# a=0;b=3; num=b-a
# for xh in out_nz[a:b,0]:
#     z=parcel['z'].isel(xh=xh).values/1000; 
#     plt.plot(z,color='blue',label='SBZ parcel')
    
# #     ind=np.where(out_nz[:,0]==xh)[0][0]
# #     z_gust=parcel['z'].isel(time=out_nz[ind][4],xh=xh).values/1000;
# #     z_lfc=parcel['z'].isel(time=out_nz[ind][5],xh=xh).values/1000;
    
# #     plt.scatter(out_nz[ind][4],z_gust,color='red',zorder=10)
# #     plt.scatter(out_nz[ind][5],z_lfc,color='purple',zorder=10)
    
# for xh in save_nz[0:num,0]:
#     z=parcel['z'].isel(xh=xh).values/1000
#     plt.plot(z,color='black',label='non-SBZ parcel')
    
# plt.title(f'comparison of {num} SBZ and non SBZ parcels')
# plt.xlabel('timestep')
# plt.ylabel('z level (m)')

# handles, labels = plt.gca().get_legend_handles_labels()
# plt.legend(handles[num-1:-(num-1)], labels[num-1:-(num-1)])

In [29]:
# #2D Plot colored by height
# parcel=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_pdata_test5.nc')
# parcel=parcel.isel(time=np.arange(0, len(parcel['time']), 30))
# var_list= [x for x in parcel]
# k=out_nz[0][0];interval=1
# x=parcel[var_list[int(0)]].isel(xh=k).values
# y=parcel[var_list[int(1)]].isel(xh=k).values
# z=parcel[var_list[int(2)]].isel(xh=k).values
# plt.scatter(x[::interval],y[::interval], c=z[::interval], cmap='viridis', marker='o')
# plt.plot(x[::interval], y[::interval], linestyle='-', color='k', alpha=0.5)
# plt.colorbar(label='z height (m)')

In [30]:
# #3D Plot colored by time
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
# from matplotlib.animation import FuncAnimation

# # Sample data
# k=out_nz[0][0];interval=1
# x=parcel[var_list[int(0)]].isel(xh=k).values
# y=parcel[var_list[int(1)]].isel(xh=k).values
# z=parcel[var_list[int(2)]].isel(xh=k).values
# var_list= [x for x in parcel]

# # Create a 3D plot
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')

# # Plot the data points
# sc=ax.scatter(x[::interval], y[::interval], z[::interval], c=np.array(range(0,len(x[::interval]))), marker='o')
# ax.plot(x[::interval], y[::interval], z[::interval], linestyle='-', color='k', alpha=0.5)
# fig.colorbar(sc,label='time',shrink=0.8)

# # Set labels for the axes
# ax.set_xlabel('X Axis')
# ax.set_ylabel('Y Axis')
# ax.set_zlabel('Z Axis')
# # Show the plot
# plt.show()

In [31]:
# #checking if any parcels crossed boundaries
# #x boundary
# print('x boundary')
# lst=[]
# for xh in range(1000):
#     back_x= parcel['x'].isel(xh=xh).values/1000
#     back_u= parcel['u'].isel(xh=xh).values/1000
#     if np.any(back_x+back_u>np.max(data['xf'].values))==True: lst.append(xh)
# where=np.where(np.array([item in lst for item in out_nz[:,0]])==True)    
# print(where)

# if np.any(where):
#     row=where[0][0]
#     back_x= parcel['x'].isel(xh=out_nz[row,0]).values/1000
#     back_u= parcel['u'].isel(xh=out_nz[row,0]).values/1000
#     print(np.any(back_x+back_u>np.max(data['xf'].values)))

#     print('crossed boundary at ' + str(np.where(back_x+back_u>np.max(data['xf'].values))))
#     print(out_nz[10])

# #y boundary
# print('y boundary')
# lst=[]
# for xh in range(1000):
#     back_y= parcel['y'].isel(xh=xh).values/1000
#     back_v= parcel['v'].isel(xh=xh).values/1000
#     if np.any(back_y+back_v>np.max(data['yf'].values))==True: lst.append(xh)
# where=np.where(np.array([item in lst for item in out_nz[:,0]])==True)    
# print(where)

# if np.any(where):
#     row=where[0][0]
#     back_y= parcel['y'].isel(xh=out_nz[row,0]).values/1000
#     back_v= parcel['v'].isel(xh=out_nz[row,0]).values/1000
#     print(np.any(back_y+back_v>np.max(data['yf'].values)))

#     print('crossed boundary at ' + str(np.where(back_y+back_v>np.max(data['yf'].values))))
#     print(out_nz[10])
# else: print('y boundary did not cross')

# #z boundary
# print('z boundary')
# lst=[]
# for xh in range(1000):
#     back_z= parcel['z'].isel(xh=xh).values/1000
#     back_w= parcel['w'].isel(xh=xh).values/1000
#     if np.any(back_z+back_w>np.max(data['zf'].values))==True: lst.append(xh)
# where=np.where(np.array([item in lst for item in out_nz[:,0]])==True)    
# print(where)

# if np.any(where):
#     row=where[0][0]
#     back_z= parcel['z'].isel(xh=out_nz[row,0]).values/1000
#     back_w= parcel['w'].isel(xh=out_nz[row,0]).values/1000
#     print(np.any(back_z+back_w>np.max(data['zf'].values)))

#     print('crossed boundary at ' + str(np.where(back_z+back_w>np.max(data['zf'].values))))
#     print(out_nz[10])
# else: print('z boundary did not cross')

In [15]:
# #2D Animation colored by height
# import matplotlib.animation as animation
# from matplotlib.animation import FuncAnimation
# k=3;interval=15
# x=parcel[var_list[int(0)]].isel(xh=k).values
# y=parcel[var_list[int(1)]].isel(xh=k).values
# z=parcel[var_list[int(2)]].isel(xh=k).values
# x=x[::interval]
# y=y[::interval]
# z=z[::interval]

# fig,ax = plt.subplots(); x_values = []; y_values = []; z_values = []; frames = []
# sc = ax.scatter([], [], c=[], cmap='viridis', marker='o',vmin=np.min(z), vmax=np.max(z)) # 
# cbar = fig.colorbar(sc, ax=ax, label='z height (m)') #

# def update(frame):
#     ax.cla()
#     x_values = x[:frame+1]
#     y_values = y[:frame+1]
#     z_values = z[:frame+1]
    
#     sc = ax.scatter(x_values, y_values, c=z_values, cmap='viridis', marker='o') #
#     ax.plot(x_values, y_values, linestyle='-', color='k', alpha=0.5) #
#     ax.set_xlim([np.min(x)-20, np.max(x)+20])
#     ax.set_ylim([np.min(y)-20, np.max(y)+20])
    
#     return sc,
# ani = FuncAnimation(fig, update, frames=len(x), interval=100, blit=True)
# ani.save('/mnt/lustre/koa/koastore/torri_group/air/particle_2d.gif', writer='pillow')
# plt.show()

In [22]:
# #3D Animation colored by time
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d import Axes3D
# from matplotlib.animation import FuncAnimation
# import numpy as np

# # Sample data
# k = 3; interval=15
# x = parcel[var_list[int(0)]].isel(xh=k).values
# y = parcel[var_list[int(1)]].isel(xh=k).values
# z = parcel[var_list[int(2)]].isel(xh=k).values
# x=x[::interval]
# y=y[::interval]
# z=z[::interval]

# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# sc = ax.scatter([], [], c=[], cmap='viridis', marker='o',lw=1,vmin=np.min(np.array(range(0,len(x)))), vmax=np.max(np.array(range(0,len(x))))) # 
# cbar = fig.colorbar(sc, ax=ax, label='time',shrink=0.8) #

# def update(frame):
#     ax.cla()
    
#     x_values = x[:frame+1]
#     y_values = y[:frame+1]
#     z_values = z[:frame+1]

#     # Plot the data points for the current frame
#     sc = ax.scatter(x_values, y_values, z_values, c=np.array(range(0,len(x_values))), marker='o', lw=1)
#     ax.plot(x_values, y_values, z_values, linestyle='-', color='k', alpha=0.5)
    
#     ax.set_xlim([np.min(x)-20, np.max(x)+20])
#     ax.set_ylim([np.min(y)-20, np.max(y)+20])
#     ax.set_zlim([np.min(z)-20, np.max(z)+20])

#     return sc,

# # Number of frames

# ani = FuncAnimation(fig, update, frames=len(x), interval=100, blit=True)

# # Save the animation
# ani.save('/mnt/lustre/koa/koastore/torri_group/air/particle_3d.gif', writer='pillow')

# plt.show()

In [ ]:
##############################
#TESTING

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import xarray as xr
# import os; import time
# # pip install h5netcdf; pip install netCDF4

# data=xr.open_dataset(dir+'cm1r20.3/run/cm1out_test7tundra-7_062217.nc') 
# parcel=xr.open_dataset(dir+'cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') 
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# parcel=parcel.isel(time=times.astype(int)) 
# var_list= [x for x in parcel]
# for i, var_name in enumerate(var_list, start=1):
#     print(f"({i-1}) {var_name}")

In [ ]:
# #code for particle ascending above LFC (legrangian) 
# # if 'data4' not in locals(): data4=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_test4.nc')
# if 'data5' not in locals(): data5=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_test3.nc')
# if 'parcel' not in locals(): 
#     parcel=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_pdata_test3.nc')
#     parcel=parcel.isel(time=np.arange(0, len(parcel['time']), 5))

# var_list= [x for x in parcel]
# # for i, var_name in enumerate(var_list, start=1):
# #     print(f"({i-1}) {var_name}")

# def parcel_info(parcel,data,t):
#     x=parcel['x'].isel(time=t).values/1000; xf=data['xf'].values; which_x=np.searchsorted(xf,x)-1 #finds which x layer parcel in
#     y=parcel['y'].isel(time=t).values/1000; yf=data['yf'].values; which_y=np.searchsorted(yf,y)-1 #finds which y layer parcel in
#     z=data['zf'].values.max()-parcel['z'].isel(time=t).values/1000; zf=data['zf'].values; which_z=np.searchsorted(zf,z)-1

#     #Find LFC levels
#     lfc=data['lfc'].isel(time=t).values/1000; zlev=data['zf'].values; which_lfc=np.searchsorted(zlev,lfc) #finds which z layer parcel in
#     tz=np.zeros((data['yh'].shape[0],data['xh'].shape[0]))
#     tlfc=np.zeros((data['yh'].shape[0],data['xh'].shape[0]))
#     pinfo=np.zeros((1000,5),dtype='int') #num,x,y,z, LFC
#     pinfo_plot=np.zeros_like(which_lfc)
#     for tr in range(0,1000): #tracer number
#         pinfo[tr,0]=tr+1; pinfo[tr,1]=which_x[tr]; pinfo[tr,2]=which_y[tr]; pinfo[tr,3]=which_z[tr]; pinfo[tr,4]=which_lfc[which_y[tr],which_x[tr]] #marks tracer number at individual columns
#         pinfo_plot[which_y[tr],which_x[tr]]=tr+1 #for plotting
        
#         tlfc[which_y[tr],which_x[tr]]=which_lfc[which_y[tr],which_x[tr]] #finds the lfc value at each tracer locaiton 
#         tz[which_y[tr],which_x[tr]]= which_z[tr] #marks tracer z location at individual columns
#     return(pinfo,pinfo_plot,tlfc,tz)
    
# t=100#900
# [pinfo,pinfo_plot,tlfc,tz]=parcel_info(parcel,data5,t)
# # plt.contourf(tz>tlfc)
# plt.contourf(pinfo_plot)
# plt.title(f'Particles Above LFC t={t}')

In [ ]:
# # parcel locations locations
# data5=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_test3.nc').isel(time=slice(20))
# parcel=xr.open_dataset('/mnt/lustre/koa/koastore/torri_group/air/cm1r20.3/run/cm1out_pdata_test3.nc').isel(time=slice(20))
# import matplotlib.animation as animation
# from matplotlib.animation import FuncAnimation
# fig,ax = plt.subplots()

# def update(frame):
#     ax.cla()
#     t=frame
#     z=parcel['z'].isel(time=t).values/1000
#     which_parcel=np.where(np.greater(z,0.5)*np.less(z,0.6)*1)[0] #between 0.5 and 0.6 km
#     which_parcel=np.where(np.greater(z,0)*np.less(z,3)*1)[0] #between 0 and 3 km
#     x=parcel['x'].isel(time=t,xh=which_parcel)/1000
#     y=parcel['y'].isel(time=t,xh=which_parcel)/1000
#     sc=plt.scatter(x,y,s=1)

#     plt.title(f'Location of all Particles t={frame}')
#     return sc,

# ani = FuncAnimation(fig, update, frames=range(0, len(data5['time']), 1), interval=300, blit=True)
# ani.save('/mnt/lustre/koa/koastore/torri_group/air/particle_locations.gif', writer='pillow')